In [95]:
%load_ext autoreload
%autoreload 2
import os
import json
import re
import pandas as pd
from thread_gpt_suite.thread_gpt_mp_handler import ThreadGPTMPHandler


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [96]:
def run_llm(system_prompt, prompts, prompt_to_code, out_path="/homes/tchakrab/scratch/ModelingCulturalValueShifts/results/CoT/raw_results.jsonl",num_workers=350):
    print("Running LLM...")
    handler = ThreadGPTMPHandler(
        api_key=os.environ.get("OPENAI_API_KEY"),
        num_worker=num_workers,
    )

    batch = [
        {
            "questions": [prompt],
            "model_name": "gpt-4.1-mini",
            "task_desc": system_prompt,
        }
        for prompt in prompts
    ]

    handler.add_batch(batch)
    results = handler.process(rerun_on_error=True)

    results_json = {}
    for result in results:
        for prompt, response in result.items():
            code_info = prompt_to_code[prompt]
            person_id = code_info['person_id']
            question_codes = code_info['question_codes']
            batch_key = f"{person_id}_batch_" + ",".join(map(str, question_codes))


            results_json[batch_key] = {
                "person_id": code_info["person_id"],
                "question_codes": code_info["question_codes"],
                "response": response,
            }

    with open(out_path, "w") as f:
        json.dump(results_json, f, indent=4)

    return out_path

In [ ]:
# Pick a country and figure out the questions that had a significant change
# country = TWN
country = "IND"